### Part 1.1 Data Analysis

In [2]:
import csv
import statistics
from collections import Counter
 
with open("annotations.csv") as f:
    rows = list(csv.DictReader(f))
 
datasets = ["fineweb", "finepdfs", "c4", "redpajama"]
print(f"Loaded {len(rows)} rows")


Loaded 124 rows


In [3]:
# quality score
qs = [int(r["quality"]) for r in rows]
 
print("mean:", round(sum(qs)/len(qs), 2))
print("median:", statistics.median(qs))
print("score <= 3:", sum(1 for q in qs if q <= 3), f"({sum(1 for q in qs if q <= 3)/len(qs)*100:.1f}%)")
print("score >= 7:", sum(1 for q in qs if q >= 7), f"({sum(1 for q in qs if q >= 7)/len(qs)*100:.1f}%)")
print("distribution:", dict(sorted(Counter(qs).items())))

mean: 4.48
median: 5.0
score <= 3: 31 (25.0%)
score >= 7: 10 (8.1%)
distribution: {1: 7, 2: 8, 3: 16, 4: 27, 5: 30, 6: 26, 7: 9, 8: 1}


In [4]:
# quality per dataset
for ds in datasets:
    ds_qs = [int(r["quality"]) for r in rows if r["dataset"] == ds]
    print(f"{ds}: {sum(ds_qs)/len(ds_qs):.2f}")


fineweb: 4.71
finepdfs: 4.26
c4: 4.29
redpajama: 4.68


In [5]:
# cleaning
total = len(rows)
c = Counter(r["correctly_cleaned"] for r in rows)
for k in ["yes", "partial", "no"]:
    print(f"{k}: {c[k]} ({c[k]/total*100:.1f}%)")


yes: 75 (60.5%)
partial: 40 (32.3%)
no: 9 (7.3%)


In [6]:
# cleaning per dataset
for ds in datasets:
    ds_rows = [r for r in rows if r["dataset"] == ds]
    c = Counter(r["correctly_cleaned"] for r in ds_rows)
    print(f"{ds}: yes={c.get('yes',0)} partial={c.get('partial',0)} no={c.get('no',0)}")

fineweb: yes=15 partial=12 no=4
finepdfs: yes=15 partial=15 no=1
c4: yes=24 partial=4 no=3
redpajama: yes=21 partial=9 no=1


In [7]:
private = [r for r in rows if r["private"] == "yes"]
harmful = [r for r in rows if r["harmful"] == "yes"]

print(f"private: {len(private)}")
for r in private:
    print(f"  {r['dataset']} {r['doc_index']}: {r['text_preview'][:80]}")

print(f"\nharmful: {len(harmful)}")
for r in harmful:
    print(f"  {r['dataset']} {r['doc_index']}: {r['text_preview'][:80]}")


private: 9
  finepdfs 12: ## 888 Leagues  **Sunday, September 24**  **Group 17**  **Tables 33, 34**  | Rat
  finepdfs 20: LIST OF CANDIDATES SUCCESSFULLY COMPLETED THE  COMPETENCY BASED EVALUATION HELD 
  finepdfs 21:   | to | tČí | Jméno          | Čas     | |----|-----|----------------|---------
  finepdfs 22: Issue:  January 2023  † Message form Father Kostandin  Vellezer dhe Motra ne Kri
  finepdfs 24: COAST GUARD HEADQUARTERS DIRECTORATE OF RECRUITMENT  CALL LETTERS FOR FINAL MEDI
  finepdfs 26: NORTH WEST NOORDWES  PROVINCIAL GAZETTE PROVINSIALE KOERANT  Vol. 261 MAHIKENG 1
  finepdfs 27: 二Ｏ一八年十一月  November 2018  Issue 59 期  稜聲音樂 Prism Music  透過光線、散發異彩。  源自三藩市，成立於2006
  redpajama 18: UCL CBT March Research Newsletter By Nikhil VadgamaMarch 14, 2020Research Newsle
  redpajama 19: Committee / Managers Hurling Teams Individual Players Total Honours Club Raffle 

harmful: 2
  c4 27: Vashikaran Specialist Chandigarh is a black magic study of attraction, by which 
  redpajama 26: Choos

In [32]:
# topics
topics = Counter(r["topic"] for r in rows)
for t, n in topics.most_common():
    print(f"{t}: {n} ({n/total*100:.1f}%)")

blog post: 39 (31.5%)
news article: 24 (19.4%)
legal/government: 16 (12.9%)
product page: 11 (8.9%)
academic/scientific: 11 (8.9%)
forum post: 10 (8.1%)
technical documentation: 6 (4.8%)
error/stub: 4 (3.2%)
spam/SEO: 2 (1.6%)
product review: 1 (0.8%)


In [33]:
# topics per dataset
for ds in datasets:
    ds_rows = [r for r in rows if r["dataset"] == ds]
    t = Counter(r["topic"] for r in ds_rows)
    print(f"\n{ds}")
    for topic, n in t.most_common():
        print(f"  {topic}: {n}")


fineweb
  news article: 13
  blog post: 8
  forum post: 4
  product page: 3
  error/stub: 2
  product review: 1

finepdfs
  legal/government: 15
  academic/scientific: 6
  technical documentation: 5
  news article: 2
  blog post: 2
  product page: 1

c4
  blog post: 13
  forum post: 5
  product page: 5
  news article: 4
  technical documentation: 1
  error/stub: 1
  academic/scientific: 1
  spam/SEO: 1

redpajama
  blog post: 16
  news article: 5
  academic/scientific: 4
  product page: 2
  legal/government: 1
  forum post: 1
  error/stub: 1
  spam/SEO: 1


In [34]:
# document length
for ds in datasets:
    ds_rows = [r for r in rows if r["dataset"] == ds]
    chars = [int(r["num_chars"]) for r in ds_rows]
    sents = [int(r["num_sentences"]) for r in ds_rows]
    print(f"{ds}: avg chars={sum(chars)/len(chars):.0f}, avg sentences={sum(sents)/len(sents):.1f}")


fineweb: avg chars=1867, avg sentences=15.4
finepdfs: avg chars=55159, avg sentences=212.6
c4: avg chars=1898, avg sentences=18.6
redpajama: avg chars=4176, avg sentences=35.8


In [35]:
private = [r for r in rows if r["private"] == "yes"]
harmful = [r for r in rows if r["harmful"] == "yes"]

print(f"private=yes: {len(private)} ({len(private)/total*100:.1f}%)")
for r in private:
    print(f"{r['dataset']:12s} idx={r['doc_index']:>3}: {r['text_preview'][:80]}")

print()
print(f"harmful=yes: {len(harmful)} ({len(harmful)/total*100:.1f}%)")
for r in harmful:
    print(f"{r['dataset']:12s} idx={r['doc_index']:>3}: {r['text_preview'][:80]}")

private=yes: 9 (7.3%)
finepdfs     idx= 12: ## 888 Leagues  **Sunday, September 24**  **Group 17**  **Tables 33, 34**  | Rat
finepdfs     idx= 20: LIST OF CANDIDATES SUCCESSFULLY COMPLETED THE  COMPETENCY BASED EVALUATION HELD 
finepdfs     idx= 21:   | to | tČí | Jméno          | Čas     | |----|-----|----------------|---------
finepdfs     idx= 22: Issue:  January 2023  † Message form Father Kostandin  Vellezer dhe Motra ne Kri
finepdfs     idx= 24: COAST GUARD HEADQUARTERS DIRECTORATE OF RECRUITMENT  CALL LETTERS FOR FINAL MEDI
finepdfs     idx= 26: NORTH WEST NOORDWES  PROVINCIAL GAZETTE PROVINSIALE KOERANT  Vol. 261 MAHIKENG 1
finepdfs     idx= 27: 二Ｏ一八年十一月  November 2018  Issue 59 期  稜聲音樂 Prism Music  透過光線、散發異彩。  源自三藩市，成立於2006
redpajama    idx= 18: UCL CBT March Research Newsletter By Nikhil VadgamaMarch 14, 2020Research Newsle
redpajama    idx= 19: Committee / Managers Hurling Teams Individual Players Total Honours Club Raffle 

harmful=yes: 2 (1.6%)
c4           idx= 27: Vashik

In [36]:
print("Private and harmful breakdown per dataset:")
for ds in datasets:
    ds_rows = [r for r in rows if r["dataset"] == ds]
    n_priv = sum(1 for r in ds_rows if r["private"] == "yes")
    n_harm = sum(1 for r in ds_rows if r["harmful"] == "yes")
    print(f"  {ds:12s}: private={n_priv}, harmful={n_harm}")

Private and harmful breakdown per dataset:
  fineweb     : private=0, harmful=0
  finepdfs    : private=7, harmful=0
  c4          : private=0, harmful=1
  redpajama   : private=2, harmful=1


### Part 1.2 OpenAI Privacy Filter

In [ ]:
pip install --upgrade transformers accelerate -q


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import pipeline
import json
import csv

In [ ]:
classifier = pipeline(
    task="token-classification",
    model="openai/privacy-filter",
)

config.json:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.80G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

In [ ]:
texts = {}
with open("samples_raw.jsonl") as f:
    for line in f:
        r = json.loads(line.strip())
        texts[(r["dataset"], r["doc_index"])] = r.get("text", "")

print(len(texts), "samples loaded")

124 samples loaded


In [ ]:
def make_serializable(spans):
    return [
        {k: float(v) if hasattr(v, 'item') else v for k, v in s.items()}
        for s in spans
    ]

results = {}
for (ds, idx), text in texts.items():
    out = classifier(text[:10000])
    results[f"{ds}_{idx}"] = make_serializable(out)

with open("privacy_filter_results.json", "w") as f:
    json.dump(results, f, indent=2)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
annotations = {}
with open("annotations.csv") as f:
    for row in csv.DictReader(f):
        annotations[(row["dataset"], int(row["doc_index"]))] = row

with open("privacy_filter_results.json") as f:
    raw_results = json.load(f)

results = {}
for key, spans in raw_results.items():
    ds, idx = key.rsplit("_", 1)
    results[(ds, int(idx))] = spans

In [ ]:
false_positives = []
false_negatives = []

for (ds, idx), spans in results.items():
    ann = annotations.get((ds, idx))
    if not ann:
        continue

    manual_private = ann["private"]
    model_flagged = len(spans) > 0

    if manual_private == "no" and model_flagged:
        false_positives.append({"dataset": ds, "doc_index": idx, "spans": spans})

    if manual_private == "yes" and not model_flagged:
        false_negatives.append({"dataset": ds, "doc_index": idx})

print(f"false positives: {len(false_positives)}")
print(f"false negatives: {len(false_negatives)}")

false positives: 49
false negatives: 0


In [ ]:
# aggregate spans into readable names per document
for r in false_positives:
    words = []
    current = []
    for s in r["spans"]:
        tag = s["entity"].split("-")[0]  # B, I, E, S, O
        token = s["word"].replace("Ġ", " ").strip()
        if tag in ("B", "S"):
            current = [token]
        elif tag == "I":
            current.append(token)
        elif tag == "E":
            current.append(token)
            words.append("".join(current))
            current = []
        if tag == "S":
            words.append(token)
    
    entity_type = r["spans"][0]["entity"].split("-")[1] if r["spans"] else ""
    print(f"{r['dataset']} idx={r['doc_index']} [{entity_type}]: {words}")

fineweb idx=0 [private_person]: ['Chloe', 'Chloeaniel', '/Jen-Jen', 'Sami']
fineweb idx=4 [private_person]: ['PatWalsh', 'Walsh', 'Kroger', 'mans', 'Wakefern', 'BrianLynch', 'Washington', 'Kroger', 'MichaelRoberson', 'Roberson', 'Roberson', 'Roberson']
fineweb idx=5 [private_person]: ['Bowman', 'MattBowman', 'MattBowman', 'Bowman', 'BowmanâĢĻs', 'PaulOlsen', 'Grinnell', 'Bowman', 'TimThornburg', 'TerryWitkowski', 'JoelHeroux', 'Bowman', 'Bowman', 'Bowman', 'GaryBowman', 'LindaBowman']
fineweb idx=6 [private_person]: ['BruceNewman']
fineweb idx=9 [private_person]: ['JillPrice']
fineweb idx=10 [private_person]: ['Dr.StephenT.Hulbert']
fineweb idx=11 [private_person]: ['SetonMotley']
fineweb idx=17 [private_person]: ['AlexHenery', 'NilesPaul', 'Henery', 'Paul', 'AdiKunalic', 'TimMarlowe']
fineweb idx=20 [private_person]: ['Andrew', 'Golota']
fineweb idx=21 [private_date]: ['Devel::NYTProf2.07_94', 'Devel-NYTProf-2.07_94.tar', 'Devel-NYTProf-2.07', 'Andreas']
fineweb idx=23 [private_person

In [ ]:
tp = fp = tn = fn = 0

for (ds, idx), spans in results.items():
    manual = annotations[(ds, idx)]["private"].strip().lower() == "yes"
    pred = len(spans) > 0

    if manual and pred:
        tp += 1
    elif not manual and pred:
        fp += 1
    elif manual and not pred:
        fn += 1
    else:
        tn += 1

print(tp, fp, fn, tn)

9 49 0 66


In [ ]:
precision = tp/(tp+fp)
recall = tp/(tp+fn)

print(f"precision={precision:.3f}, recall={recall:.3f}")

precision=0.155, recall=1.000
